# Eve-3-SABER-0.5B — Pre-tokenized Training

Run cells in order. Walk away.

1. **Setup**: git pull, install deps, login
2. **Recover checkpoint** (optional): Downloads Stage 1 weights from HuggingFace if no local checkpoint exists
3. **Pre-tokenize**: Download all datasets to disk, tokenize to .bin files (~1-2h)
4. **Train**: Reads from .bin files via mmap. Auto-resumes. ~1-2s/step.

If the pod dies, spin up a new one and run all cells again — checkpoint recovers from HF, pre-tokenized data re-downloads, training resumes.

**Volume**: 250 GB minimum.

In [ ]:
# ============================================================
# CELL 1: SETUP
# ============================================================
import os, subprocess

WORKSPACE = os.environ.get('EVE_WORKSPACE_ROOT', '/workspace')
REPO_DIR = f'{WORKSPACE}/Eve-0.5B-saber'
REPO_URL = 'https://github.com/anthony-maio/Eve-0.5B-saber.git'

# Clone or update repo
if os.path.exists(REPO_DIR):
    print('Updating repo...')
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

# Install deps (NO torch on RunPod — it ships pre-installed with CUDA)
subprocess.run(
    'pip install -q transformers accelerate datasets safetensors '
    'tokenizers wandb numpy tqdm huggingface_hub zstandard'.split(),
    check=True,
)

# HF login
hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('HF: logged in')

# WandB login
wandb_key = os.environ.get('WANDB_API_KEY', '')
if wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    print('WandB: logged in')

os.chdir(REPO_DIR)
print(f'\n✅ Ready in {REPO_DIR}')

In [ ]:
# ============================================================
# CELL 2: RECOVER CHECKPOINT FROM HUGGINGFACE (if needed)
# ============================================================
# Downloads Stage 1 weights from HF if no local checkpoint exists.
# Optimizer state is lost (Adam re-warms in ~100-500 steps, minor).
# Safe to re-run — skips if checkpoint already exists locally.

import os, sys, json, torch
from pathlib import Path

os.chdir('/workspace/Eve-0.5B-saber')
sys.path.insert(0, '.')

CKPT_DIR = Path('/workspace/eve-saber-05b/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

latest_file = CKPT_DIR / 'latest.txt'

if latest_file.exists():
    ckpt_path = Path(latest_file.read_text().strip())
    if ckpt_path.exists() and (ckpt_path / 'model.pt').exists():
        with open(ckpt_path / 'meta.json') as f:
            meta = json.load(f)
        print(f'✅ Local checkpoint found: step {meta["step"]:,}, {meta["tokens_seen"]/1e9:.2f}B tokens')
        print('   Skipping HF download.')
    else:
        latest_file.unlink()  # stale pointer, will re-download
        print('⚠️  Stale checkpoint pointer, will download from HF...')

if not latest_file.exists():
    HF_REPO = 'anthonym21/eve-3-0.5b-saber'

    print(f'Downloading checkpoint from {HF_REPO}...')
    from configuration_saber import SABERConfig
    from modeling_saber import SABERForCausalLM
    from huggingface_hub import hf_hub_download

    # Download model files from HF
    from transformers import AutoModelForCausalLM
    try:
        mdl = AutoModelForCausalLM.from_pretrained(HF_REPO, trust_remote_code=True)
    except Exception as e:
        print(f'AutoModel failed ({e}), trying manual load...')
        cfg = SABERConfig(
            vocab_size=50257, d_model=1536, n_heads=12, head_dim=128,
            n_layers=20, d_ff=2164, max_position_embeddings=2048,
            d_exp=192, d_anchor=96, n_anchors=64,
            curiosity_coeff=0.01, tie_word_embeddings=True,
        )
        mdl = SABERForCausalLM(cfg)
        # Download safetensors/bin from HF
        from huggingface_hub import snapshot_download
        dl_dir = snapshot_download(HF_REPO)
        from safetensors.torch import load_file
        try:
            state_dict = load_file(f'{dl_dir}/model.safetensors')
        except FileNotFoundError:
            state_dict = torch.load(f'{dl_dir}/pytorch_model.bin', map_location='cpu')
        mdl.load_state_dict(state_dict, strict=False)

    # Figure out what step this checkpoint was from
    # Try to read from HF model card or default to 9536 (end of Stage 1 session)
    step = 9536
    tokens_seen = step * 524288  # ACTUAL_BATCH_TOKENS
    stage_idx = 0
    try:
        readme_path = hf_hub_download(HF_REPO, 'README.md')
        with open(readme_path) as f:
            content = f.read()
        import re
        step_match = re.search(r'Step.*?(\d[\d,]+)', content)
        tokens_match = re.search(r'Tokens.*?([\d.]+)B', content)
        if step_match:
            step = int(step_match.group(1).replace(',', ''))
        if tokens_match:
            tokens_seen = int(float(tokens_match.group(1)) * 1e9)
    except Exception:
        pass

    # Save in workspace checkpoint format
    ckpt_path = CKPT_DIR / f'step_{step:08d}'
    ckpt_path.mkdir(parents=True, exist_ok=True)
    torch.save(mdl.state_dict(), ckpt_path / 'model.pt')

    meta = {
        'step': step,
        'tokens_seen': tokens_seen,
        'stage_idx': stage_idx,
        'wandb_run_id': None,
        'total_target': 50_000_000_000,
        'recovered_from': HF_REPO,
        'note': 'Optimizer state not available (re-warms in ~100-500 steps)',
    }
    with open(ckpt_path / 'meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    with open(latest_file, 'w') as f:
        f.write(str(ckpt_path))

    del mdl
    print(f'\n✅ Recovered from HF: step {step:,}, {tokens_seen/1e9:.2f}B tokens')
    print(f'   Saved to {ckpt_path}')
    print(f'   ⚠️  No optimizer state — Adam will re-warm over ~100-500 steps')

In [ ]:
# ============================================================
# CELL 2: PRE-TOKENIZE ALL DATASETS (~1-2h, resumable)
# ============================================================
# Downloads full parquet files (parallel, fast), tokenizes to .bin.
# Falls back to streaming for huge datasets (fineweb-edu, dclm).
# Safe to re-run — skips already-completed sources.
# Output: /workspace/eve-saber-05b/tokens/*.bin (~120GB total)

!python pretokenize.py

In [ ]:
# ============================================================
# CELL 3: TRAIN (auto-resumes, reads from .bin files)
# ============================================================
# Detects .bin files automatically → mmap DataLoader (~1-2s/step)
# Falls back to streaming if .bin files are missing.
# Re-run this cell after kernel restarts — it resumes from checkpoint.
# Checkpoints every 500 steps to /workspace/eve-saber-05b/checkpoints/

!cd /workspace/Eve-0.5B-saber && python train_full.py

In [ ]:
# ============================================================
# CELL 4: UPLOAD CHECKPOINT TO HUGGINGFACE (run manually)
# ============================================================
import os, sys, torch, json, shutil
from pathlib import Path

os.chdir('/workspace/Eve-0.5B-saber')
sys.path.insert(0, '.')

from huggingface_hub import HfApi
from transformers import GPT2TokenizerFast
from configuration_saber import SABERConfig
from modeling_saber import SABERForCausalLM

CKPT_DIR = Path('/workspace/eve-saber-05b/checkpoints')
EXPORT_DIR = Path('/workspace/eve-saber-05b/hf-upload')
REPO_DIR = Path('/workspace/Eve-0.5B-saber')
HF_REPO = 'anthonym21/eve-3-0.5b-saber'

ckpt_path = Path((CKPT_DIR / 'latest.txt').read_text().strip())
with open(ckpt_path / 'meta.json') as f:
    meta = json.load(f)
print(f'Checkpoint: {ckpt_path.name} | Step {meta["step"]:,} | {meta["tokens_seen"]/1e9:.2f}B tokens')

cfg = SABERConfig(
    vocab_size=50257, d_model=1536, n_heads=12, head_dim=128,
    n_layers=20, d_ff=2164, max_position_embeddings=2048,
    d_exp=192, d_anchor=96, n_anchors=64,
    curiosity_coeff=0.01, tie_word_embeddings=True,
)
mdl = SABERForCausalLM(cfg)
mdl.load_state_dict(torch.load(ckpt_path / 'model.pt', map_location='cpu'))
tok = GPT2TokenizerFast.from_pretrained('gpt2')
tok.pad_token = tok.eos_token

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)
mdl.save_pretrained(EXPORT_DIR)
tok.save_pretrained(EXPORT_DIR)
cfg.save_pretrained(EXPORT_DIR)
for f in ['configuration_saber.py', 'modeling_saber.py']:
    shutil.copy(REPO_DIR / f, EXPORT_DIR / f)

tokens_b = meta['tokens_seen'] / 1e9
stage_names = {0: 'Foundation', 1: 'Specialization', 2: 'Anneal'}
with open(EXPORT_DIR / 'README.md', 'w') as f:
    f.write(f"""---\nlicense: apache-2.0\ntags: [saber, pretrained, custom-architecture]\n---\n# Eve-3-SABER-0.5B\n500M param decoder-only LM. Tokens: {tokens_b:.1f}B/50B. Stage: {stage_names.get(meta['stage_idx'], '?')}.\n```python\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nmodel = AutoModelForCausalLM.from_pretrained("anthonym21/eve-3-0.5b-saber", trust_remote_code=True)\ntokenizer = AutoTokenizer.from_pretrained("anthonym21/eve-3-0.5b-saber")\n```\n""")

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=str(EXPORT_DIR), repo_id=HF_REPO,
    commit_message=f'Step {meta["step"]:,}, {tokens_b:.1f}B tokens ({stage_names.get(meta["stage_idx"], "?")})')
del mdl
print(f'\n✅ https://huggingface.co/{HF_REPO}')